# EDA for OilSlick Sentinel-1 GeoTIFF Data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio

from IPython.display import display
from scipy.stats import zscore

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'viridis'

## Load metadata

The dataset is situated in `waterbench_data/data/OilSlick`. The metadata table stores relative paths for the GeoTIFFs and the images used.

In [ ]:
BASE_DIR = Path('../10_waterbench_data/data/OilSlick').resolve()
METADATA_PATH = BASE_DIR / 'metadata.csv'

print('Using dataset root:', BASE_DIR)

df = pd.read_csv(METADATA_PATH)
df['label_name'] = df['label'].map({0: 'negative', 1: 'positive'})
df[['sample_id', 'label_name', 'subcategory', 'image_path', 'n_bands']].head()

In [ ]:
print(f'Samples: {len(df):,}')
print('Columns:', ', '.join(df.columns))
print()
display(df[['label_name', 'subcategory','nodata_fraction', 'is_valid', 'n_bands']].value_counts().rename('count').reset_index().head(20))

## GeoTIFF helpers

The GeoTIFFs are multi-band images. The VV and VH bands are the first two bands, and these need to be normalized using z-score to be visualized properly.

In [ ]:
def get_img_path(relative_path: str) -> Path:
    p = Path(relative_path)
    filename = p.name if p.stem.endswith('_s1') else f"{p.stem}_s1{p.suffix}"
    return BASE_DIR / 'images_s1' / filename

def plot_grayscale(sample_row, ax, title):
    path = get_img_path(sample_row['image_path'])
    
    with rasterio.open(path) as src:
        image = src.read(1)
    
    image_z = zscore(image, axis=None, nan_policy='omit')
    
    if not np.isnan(image_z).all():
        image = np.clip(image_z, -3, 3)
        image = (image + 3) / 6
    else:
        image = np.zeros_like(image)

    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(title)

## Metadata Distributions

Analysis of the metadata distributions (labels, subcategories, cloud cover, reflectance, and spatial distribution).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.countplot(data=df, x='label_name', ax=axes[0], order=['negative', 'positive'])
axes[0].set_title('Label distribution')
axes[0].set_xlabel('')

sns.countplot(data=df, x='subcategory', ax=axes[1], order=df['subcategory'].value_counts().index)
axes[1].set_title('Subcategory distribution')
axes[1].tick_params(axis='x', rotation=45)

sns.histplot(data=df, x='cloud_cover', bins=30, ax=axes[2])
axes[2].set_title('Cloud cover distribution')

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.boxplot(data=df, x='label_name', y='reflectance_mean', ax=axes[0])
axes[0].set_title('Reflectance Mean by Class')

sns.histplot(data=df, x='reflectance_mean', bins=30, ax=axes[1])
axes[1].set_title('Reflectance Mean')

sns.scatterplot(data=df.sample(min(len(df), 400), random_state=2026), x='center_lon', y='center_lat', hue='label_name', alpha=0.7, ax=axes[2])
axes[2].set_title('Spatial Distribution (400 Samples)')
axes[2].legend(title='')

plt.tight_layout()

## GeoTIFF examples

Positive and negative examples of the original GeoTIFFs are shown below.

In [ ]:
all_samples = df[df['image_path'].apply(lambda img_path: get_img_path(img_path).exists())]

pos_samples = all_samples[all_samples['label'] == 1].head(2)
neg_samples = all_samples[all_samples['label'] == 0].head(2)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, (_, row) in zip(axes[0], pos_samples.iterrows()):
    plot_grayscale(row, ax=ax, title=f"Positive | {row['sample_id']}")

for ax, (_, row) in zip(axes[1], neg_samples.iterrows()):
    plot_grayscale(row, ax=ax, title=f"Negative | {row['sample_id']}")

plt.tight_layout()
plt.show()